# Prerequisites: `.env` file

Load DataRobot credentials into your environment before you run the rest of this notebook. Create an `.env` file next to this notebook (or export variables in your shell) with at least:

```bash
DATAROBOT_ENDPOINT=YOUR_DATAROBOT_ENDPOINT
DATAROBOT_API_TOKEN=YOUR_DATAROBOT_API_TOKEN
```

Replace the placeholders with your API endpoint and token. For how routing uses these values, see [LLM configuration](llm.md).

In [ ]:
# -------------------
# 0. Load dotenv and install dependencies.
# -------------------
%load_ext dotenv
%dotenv

# Uncomment this if you run it externally
# !uv pip install "datarobot-genai[langgraph]>=0.15.18"

# Comment this if you run it externally
!uv pip install -e "../..[langgraph]"

!uv tool install git+https://github.com/carsongee/get-datarobot-llms.git

# Agent building blocks

## LLM

The DataRobot LLM Gateway lists the models your account can use. Run the next cell to see ids you can pass to `get_datarobot_gateway_llm`.

**Tip:** This step may take a moment depending on your network connection.

In [ ]:
# -------------------
# 1.1. List available LLM ids.
# -------------------
!dr get-llms

`datarobot_genai` exposes a unified LLM layer (adapters) for LangGraph, LlamaIndex, CrewAI, and NAT. For the DataRobot LLM Gateway, import `get_datarobot_gateway_llm` from the `llm` submodule of the integration you use (here: `datarobot_genai.langgraph.llm`).

In [ ]:
# -------------------
# 1.2. Instantiate an LLM object.
# -------------------
from datarobot_genai.langgraph.llm import get_datarobot_gateway_llm

llm = get_datarobot_gateway_llm("azure/gpt-5-nano-2025-08-07")

## Prompt

DataRobot offers an option of [Prompt management](https://docs.datarobot.com/en/docs/agentic-ai/prompt-mgmt/index.html#prompt-management) where you can store, version, and share your best system prompts for agents and LLMs. Use it to select a custom prompt for your agent.

In [ ]:
# -------------------
# 2.1. List available prompt templates in DataRobot
# -------------------
import datarobot as dr

prompt_templates = dr.genai.PromptTemplate.list()
prompt_templates

In [ ]:
# -------------------
# 2.2. Select which one you want to use and convert it to a LangGraph prompt template
# -------------------
from langchain_core.prompts import ChatPromptTemplate

prompt_template = dr.genai.PromptTemplate.get("69eb56c5cc06eff822e8439f")
prompt_template_version = prompt_template.get_latest_version()

langgraph_prompt_template = ChatPromptTemplate.from_template(prompt_template_version.to_fstring())
langgraph_prompt_template

## Tools

You can attach tools from `datarobot_genai.drtools` to your agent. This example uses the calculator helper.

**Note:** `drtools` is still evolving; not every tool is supported for standalone use outside `drmcp`.

In [ ]:
# -------------------
# 3. Register tools.
# -------------------
from datarobot_genai.drtools.calculator import calculator
from langchain.tools import tool

tools = [tool(calculator)]

## Agent

Define the graph the same way you would in plain LangGraph: same primitives (`StateGraph`, nodes, edges). That makes it straightforward to reuse or adapt existing examples.

> Define it in a factory function so LLM and tools can be redefine if necessary. This is useful if you want to orchestrate your agent using low-code configuration with `dragent`, and evaluate it with different configuration options.


In [ ]:
from langchain.agents import create_agent
from langgraph.graph import END
from langgraph.graph import START
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph


def graph_factory(llm, tools, verbose):
    # -------------------
    # 4.1. ReAct agents as nodes.
    # -------------------
    research_agent = create_agent(llm, tools, debug=verbose)
    writer_agent = create_agent(llm, [], debug=verbose)

    # -------------------
    # 4.2. Wire the graph.
    # -------------------
    graph = StateGraph(MessagesState)
    graph.add_node("research", research_agent)
    graph.add_node("writer", writer_agent)
    graph.add_edge(START, "research")
    graph.add_edge("research", "writer")
    graph.add_edge("writer", END)
    return graph

# Wrap the graph for DataRobot

Each supported integration exposes a helper that turns your graph factory and prompt template into a DataRobot agent class you can run behind AG-UI.

In [ ]:
# -------------------
# 5. Convert to a DataRobot agent class.
# -------------------
from datarobot_genai.langgraph.agent import datarobot_agent_class_from_langgraph

MyAgent = datarobot_agent_class_from_langgraph(graph_factory, langgraph_prompt_template)

# Run the agent

The next cell streams AG-UI events from `invoke` so you can inspect the run lifecycle, tool calls, and text.

In [ ]:
# -------------------
# 5. Action!
# -------------------
from datarobot_genai.core.agents.render import render_ag_ui_event

agent = MyAgent(llm=llm, tools=tools, verbose=False)

request = "Calculate 2+2 and output result immediately"

async for event, pipeline_interactions, usage_metrics in agent.invoke_single_message(request):
    rendered = render_ag_ui_event(event)
    if rendered is not None:
        print(rendered, end='')